In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add src to path
sys.path.insert(0, os.path.abspath('../src'))

from preprocessing.pipeline import preprocess_pipeline
from preprocessing.visualize import (
    plot_class_distribution,
    plot_text_length,
    plot_split_distribution
)

In [ ]:
# Paths
FAKE_PATH = '../data/Fake.csv'
TRUE_PATH = '../data/True.csv'
OUTPUT_DIR = '../data'

# Load raw data
fake = pd.read_csv(FAKE_PATH)
true = pd.read_csv(TRUE_PATH)

# Add labels
fake['label'] = 0
true['label'] = 1

# Combine
df = pd.concat([fake, true], ignore_index=True)

print(f"Original Fake rows: {len(fake)}")
print(f"Original True rows: {len(true)}")
print(f"Total rows: {len(df)}")
print(f"Empty text rows: {(df['text'].str.strip() == '').sum()}")

In [ ]:
from preprocessing.clean import clean_texts

# Clean text
df['text'] = clean_texts(df['text'].astype(str))

# Remove empty text
empty_before = (df['text'] == '').sum()
df = df[df['text'] != '']
empty_after = (df['text'] == '').sum()

cleaning_stats = {
    'empty_text_before': int(empty_before),
    'empty_text_after': int(empty_after),
    'empty_text_removed': int(empty_before - empty_after)
}

print(json.dumps(cleaning_stats, indent=2))

In [ ]:
from preprocessing.duplicates import find_duplicates, remove_duplicates, report_duplicates

# Find duplicates
duplicates = find_duplicates(df)
print(f"Exact duplicates: {len(duplicates['exact_duplicates'])}")
print(f"Conflicting labels: {len(duplicates['conflicting_labels'])}")

# Remove duplicates (keep first)
df = remove_duplicates(df, strategy='keep_first')

# Report duplicates
duplicates_report = report_duplicates(df)
print(json.dumps(duplicates_report, indent=2))

In [ ]:
# Final dataset stats
final_stats = {
    'final_total_rows': len(df),
    'final_fake_rows': len(df[df['label'] == 0]),
    'final_true_rows': len(df[df['label'] == 1]),
    'final_empty_text': (df['text'] == '').sum()
}

print(json.dumps(final_stats, indent=2))

In [ ]:
from preprocessing.split import stratified_split

# Stratified split
train, val, test = stratified_split(df, random_state=42)

split_stats = {
    'train_size': len(train),
    'val_size': len(val),
    'test_size': len(test),
    'train_class_dist': train['label'].value_counts(normalize=True).to_dict(),
    'val_class_dist': val['label'].value_counts(normalize=True).to_dict(),
    'test_class_dist': test['label'].value_counts(normalize=True).to_dict()
}

print(json.dumps(split_stats, indent=2))

In [ ]:
from preprocessing.tfidf import fit_transform_tfidf

# TF-IDF vectorization (FIT ON TRAIN ONLY)
X_train, X_val, X_test, vectorizer = fit_transform_tfidf(
    train['text'],
    val['text'],
    test['text'],
    max_features=5000
)

tfidf_stats = {
    'X_train_shape': X_train.shape,
    'X_val_shape': X_val.shape,
    'X_test_shape': X_test.shape,
    'vocabulary_size': len(vectorizer.vocabulary_),
    'max_features': vectorizer.max_features,
    'ngram_range': vectorizer.ngram_range
}

print(json.dumps(tfidf_stats, indent=2))

In [ ]:
# Save processed data
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save X
np.save(os.path.join(OUTPUT_DIR, 'X_train.npy'), X_train)
np.save(os.path.join(OUTPUT_DIR, 'X_val.npy'), X_val)
np.save(os.path.join(OUTPUT_DIR, 'X_test.npy'), X_test)

# Save y
y_train = train['label'].values
y_val = val['label'].values
y_test = test['label'].values

np.save(os.path.join(OUTPUT_DIR, 'y_train.npy'), y_train)
np.save(os.path.join(OUTPUT_DIR, 'y_val.npy'), y_val)
np.save(os.path.join(OUTPUT_DIR, 'y_test.npy'), y_test)

# Save vectorizer
joblib.dump(vectorizer, os.path.join(OUTPUT_DIR, 'tfidf_vectorizer.pkl'))

print(f"Processed data saved to: {OUTPUT_DIR}")

In [ ]:
# Generate visualizations
os.makedirs('../results/figures/preprocessing', exist_ok=True)

# Plot class distribution
plot_class_distribution(df)

# Plot text length distribution
plot_text_length(df)

# Plot split distribution
plot_split_distribution(train, val, test)

print(f"Visualizations saved to: ../results/figures/preprocessing")

In [ ]:
# Run complete pipeline
metadata = preprocess_pipeline(
    FAKE_PATH,
    TRUE_PATH,
    output_dir=OUTPUT_DIR,
    max_features=5000,
    random_state=42
)

# Save metadata
with open(os.path.join(OUTPUT_DIR, 'preprocessing_metadata.json'), 'w') as f:
    json.dump(metadata, f, indent=2)

print("Preprocessing complete!")
print(f"Metadata saved to: {os.path.join(OUTPUT_DIR, 'preprocessing_metadata.json')}")

# Validation
print("\n=== VALIDATION ===")
print(f"Empty text in final dataset: {(df['text'] == '').sum()}")
print(f"Unique labels: {df['label'].unique()}")
print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_val shape: {X_val.shape}")
print(f"y_val shape: {y_val.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")
print(f"NaN in X_train: {np.isnan(X_train.data).sum()}")
print(f"Inf in X_train: {np.isinf(X_train.data).sum()}")

print("\n=== LEAKAGE AUDIT ===")
print("TF-IDF FIT DATA = TRAIN ONLY")
print(f"Vectorizer fitted on: {len(train)} training samples")
print(f"Vectorizer transformed on: {len(val)} validation samples")
print(f"Vectorizer transformed on: {len(test)} test samples")
print("No validation or test data was used for fitting.")